In [1]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain.agents.structured_output import ToolStrategy, ProviderStrategy
from langgraph.checkpoint.memory import InMemorySaver


from pydantic import BaseModel, Field

In [2]:
from math import sqrt as mathsqrt

@tool
def sum(x1: float, x2: float) -> float:
    """evaluate simple sum of x1 and x2."""
    return x1 + x2

@tool
def product(x1: float, x2: float) -> float:
    """evaluate simple product of x1 and x2."""
    return x1*x2

@tool
def divide(x1: float, x2: float) -> float:
    """divide x1 by x2"""
    return x1/x2

@tool
def squared(x: float) -> float:
    """evaluate x squared"""
    return x**2

@tool
def sqrt(x: float) -> float:
    """evaluate sqaure root of x"""
    return mathsqrt(x)

tools = [sum, product, divide, squared, sqrt]

In [3]:
class Context(BaseModel):
    """Custom runtime context schema."""
    user_id: str


class ResponseFormat(BaseModel):
    """Responde in this specific format."""

    objetive_response: str = Field(description="A direct and objetive reponse to the query.")    
    commands_used: list[dict] = Field(description="A list of all commands used and their responses.")


# Set up memory
checkpointer = InMemorySaver()

In [4]:
llm = ChatOllama(
    model="qwen3.5:4b",
    # other params...
)

agent = create_agent(
    llm,
    tools=tools,
    system_prompt="You are a helpful assistant",
    context_schema=Context,
    response_format=ResponseFormat,
    checkpointer=checkpointer,
)

In [7]:
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "calculate the roots of  f(x) = 18*x^2 + 25*x - 37"}]},
    config=config,
    context=Context(user_id="1"),
    )

Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]


In [9]:
response['structured_response'].commands_used

[{'functionName': 'squared', 'parameters': {'x': 25}, 'response': 625},
 {'functionName': 'product',
  'parameters': {'x1': 4, 'x2': 18},
  'response': 72},
 {'functionName': 'product',
  'parameters': {'x1': 72, 'x2': 37},
  'response': 2664},
 {'functionName': 'sum',
  'parameters': {'x1': 625, 'x2': 2664},
  'response': 3289},
 {'functionName': 'divide',
  'parameters': {'x1': 3289, 'x2': 36},
  'response': 91.36111111111111},
 {'functionName': 'squared', 'parameters': {'x': 170}, 'response': 28900},
 {'functionName': 'sqrt',
  'parameters': {'x': 3289},
  'response': 57.34980383575867}]